# Build a Micro-Transformer for DNA sequence

Section 1: The Positional Encoding (The "Map")Since Transformers process all DNA bases at once, they are "bag-of-words" models unless we add position.The Logic: We use Sine and Cosine waves of different frequencies.The Code Goal: Create a class PositionalEncoding(nn.Module) that adds a unique constant vector to each DNA base embedding based on its index (1, 2, 3...).



## Section 2: The Multi-Head Attention (The "Brain")
Create `class MultiHeadAttention(nn.Module)`
### The Math: Implement the Scaled Dot-Product Attention:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
The shape of *Q, K, V* are identical tensor of *(batch_size, sequence_length, d_model)*.
  * Suppose you have 10 sequences each with length 30, then batch_size=10, sequence_length=10. 
  * If *d_model=512*, every single base in the sequence is represented by 512 different features as a vector.
### 2.1 Linear Projection
* `self.w_q = nn.Linear(d_model, d_model)` sets input and output features as `d_model`
* `Q = self.w_q(q)` calculates
Note that the dimension of Q is (batch_size, sequence_len, d_model), pytorch detects the `d_model` matches the `input_features` of `self.w_q`, so it will parallelize by batch and bases automatically.
$$\text{Output} = q \times W^T + b$$

### 2.2 The Multi-Head Split
```
Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
```
* Shape change: [batch, seq_len, 512] $\rightarrow$ [batch, num_heads, seq_len, 64]
* Why we do this: By moving num_heads to the 2nd dimension, PyTorch treats the 8 heads like a "mini-batch." This allows us to calculate attention for all 8 heads simultaneously using one fast matrix multiplication.

* The Challenge: Use `.view()` and `.transpose()` to split your embedding dimension into multiple heads (e.g., 320 dimensions into 8 heads of 40).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads # Dimension per head
        
        # Linear layers for Q, K, V
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v):
        batch_size = q.size(0)
        
        # 1. Linear transformations & split into heads
        # Shape: [batch, seq_len, heads, d_k]
        Q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 2. Scaled Dot-Product Attention
        # Formula: Softmax( (QK^T) / sqrt(d_k) ) * V
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attention = F.softmax(scores, dim=-1)
        out = torch.matmul(attention, V)

        # 3. Concatenate heads back together
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.fc_out(out)

Section 3: The Transformer Layer (The "Assembly")The Logic: Combine Attention + LayerNorm + FeedForward + Residual Connections (Add & Norm).The Code Goal: Build the EncoderLayer and then the TransformerEncoder which stacks them.
Section 4: The Genomic Task (TATA-Box Detection)The Data: Generate synthetic DNA.Positive Set: 50bp sequences containing the TATAAA motif at a specific position.Negative Set: 50bp random DNA sequences.The Tokenizer: Since you are building from scratch, use a simple character-level map: {'A': 1, 'C': 2, 'G': 3, 'T': 4, '<pad>': 0}.